# Rail Detection Deep Learning Project

In [ ]:
# importing packages
import csv
import pandas as pd 
import os 
import shutil
import tensorflow as tf 
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

## Reading the dataset 
Using pandas and shutil to move all the images in the dataset to the correct folder either "defective" or "non_defective" by first reading the csv file which came with the dataset and then finding those images and moving them.

In [ ]:
df = pd.read_csv('train _combined.csv')
df.head()

# Sökvägen till varje directory
raw_directory = 'raw_data'
defective_directory = 'data/defective'
non_defective_directory = 'data/non_defective'

# Går igenom csv filen rad för rad 
for index, row in df.iterrows(): 
    file_name = row['images']
    label = row['labels']
    
    # Var bilderna ligger
    source = os.path.join(raw_directory, file_name)
    
    if label == 'Defective':
        destination = os.path.join(defective_directory, file_name)
    else: 
        destination = os.path.join(non_defective_directory, file_name)
    
    # Kolla om bilden redan finns i destination mappen 
    if not os.path.exists(destination):    
        # kopiera bilden till den rätta mappen 
        shutil.copy(source, destination)
    else: 
        # Om den redan finns -> Gör ingenting
        pass
    
    
print("Done with moving all the pictures")

## Konstanter

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 1337

## Preprocessing 
Vi kommer inte få flera bilder direkt nu, eftersom hela processen körs "live" dvs att vi kommer lägga in hela data_augmentation som första lagret i modellen. PÅ det sättet kommer effekterna att appliceras live medan bilderna laddas upp till nätverket.

In [ ]:
# Data Augmentation 
data_augmentation = keras.Sequential(
    [
         
        layers.RandomFlip("horizontal"), 
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1), 
        layers.RandomBrightness(0.1),
        layers.Rescaling(1./255)
    ], 
    name = "data_augmentation",
)

train_ds = tf.keras.utils.image_dataset_from_directory(
    "data", # Huvud foldern
    validation_split=0.2, 
    subset= "training",
    seed = SEED, 
    image_size = IMAGE_SIZE,
    batch_size = BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "data",
    validation_split = 0.2,
    subset = "validation",
    seed = SEED, 
    image_size = IMAGE_SIZE,
    batch_size = BATCH_SIZE,
)

class_names = train_ds.class_names
print(class_names)


# Följande koden används för att visualisera bilderna efter data Augmentation för att se till att allting ser bra ut oc inga bilder förstörs

In [ ]:


# Plocka ut EN batch (32 bilder) från träningsdatan
for images, lbls in train_ds.take(1):
    frst_img = images[0]
    
    # För att få plats med två bilder bredvid varandra och jämföra dessa med varandra 
    plt.figure(figsize=(12, 6))
    
    # Plats för orginal bilden till vänster
    plt.subplot(1, 2, 1) 
    
    # Omvandlar 0-255 till 0-1 floats för plt 
    original_to_plot = frst_img.numpy() / 255.0
    original_to_plot = np.clip(original_to_plot, 0, 1)
    
    plt.imshow(original_to_plot)
    plt.title("Före: Original")
    plt.axis("off")
    
    
    # Place for the Augmentated pictures on the right side
    plt.subplot(1, 2, 2) # 1 rad, 2 kolumner, bild 2
    
    # Skickar bilden genom första layer
    augmented_image = data_augmentation(tf.expand_dims(frst_img, 0), training=True)
    
    # Normalisera bilden 
    aug_to_plot = augmented_image[0].numpy()
    aug_to_plot = np.clip(aug_to_plot, 0, 1)
    
    plt.imshow(aug_to_plot)
    plt.title("Efter: Augmenterad")
    plt.axis("off")
    
    # Sparar bilden till rapporten 
    plt.savefig('before_and_after_augmentation.png', bbox_inches='tight')
    plt.show()

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size = AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size = AUTOTUNE)

# Now we create the model 
build the simple baseline CNN from scratch.

In [ ]:
# we use the sequential type because the model is a simple layer-by-layer pipeline.
model = keras.Sequential([
    
    #This is the layer that applies augmentation during training.
    data_augmentation,

    #Then comes the first convolutional layer:
        #Conv2D learns small visual patterns, such as edges, lines, cracks and texture.
    layers.Conv2D(32, 3, activation="relu"),

    #MaxPooling reduces the images size.
    layers.MaxPooling2D(),

    #Second convolution layer. 
        #This learns more complex patterns based on the first layer's simple patterns.
    layers.Conv2D(64, 3, activation="relu"),

    #MaxPooling reduces the images size again.
    layers.MaxPooling2D(),

    # This learns even higher-level patterns, such as damaged rail shapes or larger textures.
    layers.Conv2D(128, 3, activation="relu"),

    # Reduce size again before sending features to the classifier part.
    layers.MaxPooling2D(),

    # Instead of flattening every pixel location into a huge vector,
    # this averages each feature map into one number.
    layers.GlobalAveragePooling2D(),


    # This combines the learned image features to make a classification decision.
    layers.Dense(64, activation="relu"),


    # Now using "Dropout" the model drops out randomly 30% of neurons during training to help prevent the model from memorizing the training images 

    layers.Dropout(0.3),

    # We use 1 neuron because this is binary classification.
    # Sigmoid outputs a probability between 0 and 1.
    # Example: 0.90 means the model strongly predicts class 1.
    layers.Dense(1, activation="sigmoid")
])

In [ ]:
# This helps you check the number of layers, output shapes, and trainable parameters.
model.summary()

# Next step is to compile the model 

In [ ]:
model.compile(

    #First we choose the optimizer Adam 
    optimizer="adam",

    #We choose the loss function 
    loss="binary_crossentropy",

    metrics=[

        "accuracy",

        keras.metrics.Precision(name="precision"),

        keras.metrics.Recall(name="recall")
    ]
)

In [ ]:
model.summary()

# Training the model now 


In [ ]:
EPOCHS = 25 


history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# Record accuracy, loss, precision, recall, and F1-score


In [ ]:

# Convert the training history into a table.
history_df = pd.DataFrame(history.history)

# F1-score combines precision and recall into one value.
# Formula: F1 = 2 * (precision * recall) / (precision + recall)
# We add a very small number (1e-7) to avoid division by zero.
history_df["f1_score"] = 2 * (
    history_df["precision"] * history_df["recall"]
) / (history_df["precision"] + history_df["recall"] + 1e-7)

# Validation F1-score uses validation precision and validation recall.
# This is important because validation results better show performance on unseen data.
history_df["val_f1_score"] = 2 * (
    history_df["val_precision"] * history_df["val_recall"]
) / (history_df["val_precision"] + history_df["val_recall"] + 1e-7)

# Add an epoch number column so the table is easier to read.
history_df["epoch"] = range(1, len(history_df) + 1)

# Choose the columns we want to record for the project report.
metrics_table = history_df[[
    "epoch",
    "accuracy",
    "loss",
    "precision",
    "recall",
    "f1_score",
    "val_accuracy",
    "val_loss",
    "val_precision",
    "val_recall",
    "val_f1_score"
]]

# Save the metrics table as a CSV file.
# This gives you a permanent record of the baseline CNN results.
metrics_table.to_csv("baseline_cnn_metrics.csv", index=False)

# Show the table inside the notebook.
metrics_table

# Save plots for training/validation accuracy and loss

In [ ]:


# Create one figure with two plots side by side.
plt.figure(figsize=(12, 5))

# Plot 1: training accuracy vs validation accuracy.
# If training accuracy is much higher than validation accuracy, the model may be overfitting.
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Baseline CNN Accuracy")
plt.legend()

# Plot 2: training loss vs validation loss.
# Loss tells us how wrong the model is; lower loss is better.
# If validation loss rises while training loss falls, that is another sign of overfitting.
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Baseline CNN Loss")
plt.legend()

# tight_layout makes sure labels and titles do not overlap.
plt.tight_layout()

# Save the plot for the report.
plt.savefig("baseline_cnn_accuracy_loss.png", dpi=300, bbox_inches="tight")

# Display the plot in the notebook.
plt.show()

# Print the final recorded results

In [ ]:
# Get the last row from the metrics table.
# The last row represents the final epoch of training.
final_results = metrics_table.iloc[-1]

# Print the final training and validation metrics in a readable format.
print("Final baseline CNN results")
print("Training accuracy:", final_results["accuracy"])
print("Training loss:", final_results["loss"])
print("Training precision:", final_results["precision"])
print("Training recall:", final_results["recall"])
print("Training F1-score:", final_results["f1_score"])
print("Validation accuracy:", final_results["val_accuracy"])
print("Validation loss:", final_results["val_loss"])
print("Validation precision:", final_results["val_precision"])
print("Validation recall:", final_results["val_recall"])
print("Validation F1-score:", final_results["val_f1_score"])